In [1]:
import pandas as pd
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

In [2]:
location = "india"
vehicle = "rice"
# Which result pathways this (location, vehicle) combo has. Snakemake passes
# these from config_utils.get_combo_pathways, the same function that decides
# which inputs the rule declares. A pathway the combo lacks contributes zeros,
# built from the template combo (india/rice). Never branch on whether a file
# exists: a job can start before the file is built, and a file missing by
# mistake would silently become zeros.
has_simulations = True
has_anemia_model = True
has_ntd_model = True

In [3]:
# Parameters
location = "ethiopia"
vehicle = "wheat"
has_simulations = False
has_anemia_model = False
has_ntd_model = True


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if has_simulations:
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,intervention,0,141,0
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,intervention,0,141,0
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,intervention,0,141,0
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,intervention,0,141,0
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,intervention,0,141,0
...,...,...,...,...,...,...,...,...,...,...,...
1076395,person_time,impairment,anemia,severe,95_plus,severe,1,baseline,0,54,0
1076396,person_time,impairment,anemia,severe,95_plus,severe,2,baseline,0,54,0
1076397,person_time,impairment,anemia,severe,95_plus,severe,3,baseline,0,54,0
1076398,person_time,impairment,anemia,severe,95_plus,severe,4,baseline,0,54,0


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

not_anemic    270000
mild          270000
moderate      270000
severe        270000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
intervention  1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
zero          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
intervention  1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
zero          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
intervention  1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
zero          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,8110.649607
1,Female,0.0,0.019178,not_pregnant,2,7372.711095
2,Female,0.0,0.019178,not_pregnant,3,6970.098109
3,Female,0.0,0.019178,not_pregnant,4,6250.761973
4,Female,0.0,0.019178,not_pregnant,5,5010.316721
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,1459.535125
281,Male,95.0,125.000000,not_pregnant,2,1527.072977
282,Male,95.0,125.000000,not_pregnant,3,1585.027440
283,Male,95.0,125.000000,not_pregnant,4,1677.097974


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    984432.638929
2    994873.468032
3    820614.601787
4    762092.621675
5    757672.188276
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
intervention  1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
zero          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if has_simulations:
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,intervention,0,141,0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,intervention,0,141,0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,intervention,0,141,0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,intervention,0,141,0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,intervention,0,141,0
...,...,...,...,...,...,...,...,...,...,...,...
538195,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,baseline,0,54,0
538196,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,baseline,0,54,0
538197,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,baseline,0,54,0
538198,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,baseline,0,54,0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['susceptible_to_maternal_disorders_to_maternal_disorders', 'maternal_disorders_to_recovered_from_maternal_disorders'], dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
intervention  1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
zero          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"

# NOTE: The child_scenario column currently contains only 'baseline'
# because we didn't have any interventions in the child simulation. If
# we add a child intervention that creates another scenario in this
# column, then results from different child scenarios would get added
# together in the call to aggregate_by_scenario below, so we'd need to
# change the processing code in that case.
def assert_unique_child_scenario(df):
    assert set(df.child_scenario.unique()) == {'baseline'}
    return df

if has_simulations:
    neonatal_deaths = (
        pd.read_parquet(path)
        .pipe(assert_unique_child_scenario)
        .rename(columns={"maternal_scenario": "scenario"})
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,input_draw,random_seed,value
0,deaths,cause,other_causes,other_causes,0_to_5_months,Female,1,baseline,intervention,0,81,0
1,deaths,cause,other_causes,other_causes,0_to_5_months,Female,2,baseline,intervention,0,81,0
2,deaths,cause,other_causes,other_causes,0_to_5_months,Female,3,baseline,intervention,0,81,0
3,deaths,cause,other_causes,other_causes,0_to_5_months,Female,4,baseline,intervention,0,81,0
4,deaths,cause,other_causes,other_causes,0_to_5_months,Female,5,baseline,intervention,0,81,0
...,...,...,...,...,...,...,...,...,...,...,...,...
23955,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,baseline,0,129,0
23956,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,baseline,0,129,0
23957,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,baseline,0,129,0
23958,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,baseline,0,129,0


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
intervention  1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
zero          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if has_anemia_model:
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,0,intervention
1,Female,0.0,0.019178,2,0,intervention
2,Female,0.0,0.019178,3,0,intervention
3,Female,0.0,0.019178,4,0,intervention
4,Female,0.0,0.019178,5,0,intervention
...,...,...,...,...,...,...
245,Male,95.0,125.000000,1,0,baseline
246,Male,95.0,125.000000,2,0,baseline
247,Male,95.0,125.000000,3,0,baseline
248,Male,95.0,125.000000,4,0,baseline


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
intervention  1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
zero          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
intervention  1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
zero          1                  0.0
              2                  0.0
              3                  0.0
              4                  0.0
              5                  0.0
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if has_ntd_model:
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  2198.402026
              2                  2091.183175
              3                  1903.342397
              4                  1677.268540
              5                  1323.196985
baseline      1                  2198.402026
              2                  2091.183175
              3                  1903.342397
              4                  1677.268540
              5                  1323.196985
intervention  1                  2007.263913
              2                  1730.510122
              3                  1441.187235
              4                  1420.567761
              5                  1144.642554
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)